In [ ]:
## Restructured model with germany data

In [50]:
import pickle
from cntmosaic.dataloader.restru_loaders import MergedLoader, RawLoader
import pandas as pd
import numpy as np
import xarray as xr
from cntmosaic.dataloader import CoordToColumns
from cntmosaic.models.restr_BRCfine import restr_BRCfine

In [51]:

with open('../datasets/data/polymod_germany.pkl', 'rb') as file:
    data = pickle.load(file)['contacts']


In [44]:
c = pd.read_csv('2008_Mossong_POLYMOD_contact_common.csv')
p = pd.read_csv('2008_Mossong_POLYMOD_participant_common.csv')


In [18]:
mapp = CoordToColumns(age_part='part_age', age_cnt='cnt_age_exact', id_var='part_id')
data = RawLoader(p, c, mapp)

In [19]:
data.raw_df = data.raw_df[data.raw_df.part_age < 85]
data.raw_df = data.raw_df[data.raw_df.cnt_age_exact < 85]

In [52]:
mapp = CoordToColumns(age_part='age_part', age_cnt='age_cnt', id_var='id', population_age='age', population_size='P')

In [53]:
with open('../datasets/data/polymod_germany.pkl', 'rb') as file:
    pop = pickle.load(file)['population']

In [54]:
d = MergedLoader(data, mapp, pop[pop.age < 85])

In [56]:
d.load()

In [57]:
model = restr_BRCfine(d)

new model instantiated, please check default hyperparameters


In [58]:
model.params.grid_type = 'diff-age'

In [59]:
import jax
model.run_inference_mcmc(jax.random.PRNGKey(0),
    num_samples = 1000,
    num_warmup = 500,
    num_chains = 2)

/home/yiminglin/Downloads/cntmosaic/cntmosaic/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 2 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(2)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1500/1500 [49:54<00:00,  2.00s/it, 511 steps of size 1.11e-02. acc. prob=0.94] 


Number of divergences: 4


In [60]:
# Save to a file
with open('diff-age.pkl', 'wb') as f:
    pickle.dump(model, f)


In [61]:
with open('diff-age.pkl', 'rb') as f:
    model = pickle.load(f)


In [62]:
from cntmosaic.analysis import ModelSummariserMCMC, ModelVisualiser

In [63]:
MSM = ModelSummariserMCMC(model)

In [64]:
visual = ModelVisualiser(MSM)

In [65]:
visual.plot_cint()

alt.Chart(...)

In [66]:
visual.plot_rate()

alt.Chart(...)